In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from astropy.coordinates import ICRS    
from astropy.coordinates import Galactic 
from astropy.coordinates import SkyCoord 
from mpl_toolkits.axes_grid1 import make_axes_locatable
from reproject import reproject_interp
from reproject import reproject_from_healpix
from reproject import reproject_to_healpix
import healpy as hp
from astropy.io import fits
#from qu_los_sim import make_los
import astropy.units as u
import matplotlib as mpl
import matplotlib.gridspec as gridspec
#import scipy
from astropy_healpix import HEALPix
from rmtable import RMTable

In [ ]:
def FD_moments(data, thresh_map, maxFD, RM_arr, maptype,
                M0=True, M1=True, M2=False):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use   = data[(abs(RM_arr) <= maxFD),:,]
    if maptype == 'hpx':
        RM_arr_use = np.expand_dims(RM_arr[(abs(RM_arr) <= maxFD)], axis=(1))
    if maptype == 'plat_car':
        RM_arr_use = np.expand_dims(RM_arr[(abs(RM_arr) <= maxFD)], axis=(1,2))
    print(data_use.shape)
    print(RM_arr_use.shape)

    # set any data points below PI threshold to NaN:
    data_use[data_use < thresh_map] = np.nan

    moments = {}
    if M0:
        print('Calculating M0...')
        M0_data = dFD*np.nansum(data_use, axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        print('Calculating M1...')
        M1_data = dFD*np.nansum(data_use*RM_arr_use, axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        print('Calculating M2...')
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use - M1_data)**2, axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

## Read in the data

In [ ]:
# LoTSS M1:
m1_lotss_gal_S = hp.read_map('/srv/data/lofar/lotss_50radm2/M1_ave_hpx_50_south.fits', 
                             field=[0], h='True', dtype=np.float64)[0]
print(f'LoTSS M1: {m1_lotss_gal_S.shape}')

# LoTSS binned spectra:
lotss_cube_gal_S = np.load('/srv/data/lofar/lotss_50radm2/LoTSS_cube_ave_south.npy')
fd_lotss = np.arange(-50, 50.1, 0.25)
print(f'LoTSS cube: {lotss_cube_gal_S.shape}')
print(f'LoTSS FD: {len(fd_lotss)}')

# LoTSS raw spectra:

# DRAGONS M1:
m1_dragons = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/faraday_synthesis/dragons_M1_gal.hpx.fits')[0].data
print(f'DRAGONS M1: {m1_dragons.shape}')

# DRAGONS spectra:
dragons_hdu = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/faraday_synthesis/dragons_FDF_clean_tot_Kgal.hpx.fits')
dragons_cube_gal = dragons_hdu[0].data
dragons_hdr = dragons_hdu[0].header
fd_min = dragons_hdr['CRVAL2']-(dragons_hdr['NAXIS2']-dragons_hdr['CRPIX2'])*dragons_hdr['CDELT2']
fd_max = dragons_hdr['CRVAL2']+(dragons_hdr['NAXIS2']-dragons_hdr['CRPIX2'])*dragons_hdr['CDELT2']
fd_dragons = np.arange(fd_min,fd_max+dragons_hdr['CDELT2'],dragons_hdr['CDELT2'])
print(f'DRAGONS cube: {dragons_cube_gal.shape}')
print(f'DRAGONS FD: {len(fd_dragons)}')

# GMIMS-HBN spectra:
hbn_hdu = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_hpx_FD_PI.fits')
hbn_cube_gal = hbn_hdu[0].data
hbn_hdr = hbn_hdu[0].header
fd_min = hbn_hdr['CRVAL2']
fd_max = hbn_hdr['CRVAL2']+(hbn_hdr['NAXIS2']-hbn_hdr['CRPIX2'])*hbn_hdr['CDELT2']
print(fd_min,fd_max)
fd_hbn = np.arange(fd_min,fd_max+hbn_hdr['CDELT2'],hbn_hdr['CDELT2'])
print(f'GMIMS-HBN cube: {hbn_cube_gal.shape}')
print(f'GMIMS-HBN FD: {len(fd_hbn)}')

# GMIMS-HBN M1:
thresh_map = np.ones(hbn_cube_gal.shape[1])*0.04
m1_hbn = FD_moments(hbn_cube_gal, thresh_map, 500, fd_hbn, 'hpx', M0=True, M1=True, M2=False)['M1']
print(f'GMIMS-HBN M1: {m1_hbn.shape}')

In [ ]:
print(hp.npix2nside(49152))
print(hp.npix2nside(196608))
print(hp.npix2nside(786432))

In [ ]:
cat_spiceracs = fits.open('/srv/data/spice-racs/spice-racs.dr2.2026-05-01.fits')[1].data
cat_hutschen  = RMTable.read('/srv/data/rmtable/consolidated_catalog_ver1.2.0.fits')


In [ ]:
def reproject_patch(map_in1, map_in2, map_in3, lmid, bmid, dtheta):

    dl = dtheta/np.cos(bmid*np.pi/180)
    db = dtheta
    
    l1 = lmid - dl
    l2 = lmid + dl
    b1 = bmid - db
    b2 = bmid + db

    # Make a header and WCS:
    hdu = fits.PrimaryHDU()
    hdr = hdu.header
    hdr['NAXIS'] = 2
    hdr['NAXIS1'] = 1440
    hdr['NAXIS2'] = 720
    hdr['CRVAL1'] = lmid 
    hdr['CDELT1'] = -0.25
    hdr['CRPIX1'] = 720.25
    hdr['CRVAL2'] = bmid
    hdr['CDELT2'] = 0.25
    hdr['CRPIX2'] = 360.25
    hdr['COORD'] = 'galactic'
    hdr['CTYPE1']  = 'GLON-SIN' # To make it have Mollweide projection
    hdr['CTYPE2']  = 'GLAT-SIN'
    wcs = WCS(hdr)
    
    # Reproject the Healpix map:
    map1, footprint = reproject_from_healpix((map_in1,'galactic'), hdr, nested=False)
    map2, footprint = reproject_from_healpix((map_in2,'galactic'), hdr, nested=False)
    map3, footprint = reproject_from_healpix((map_in3,'galactic'), hdr, nested=False)

    return map1, map2, map3, hdr, l1, l2, b1, b2

In [ ]:
def points_in_pixel(lmid, bmid, dtheta):

    dl = dtheta/np.cos(bmid*np.pi/180)
    db = dtheta
    
    l1 = lmid - dl
    l2 = lmid + dl
    b1 = bmid - db
    b2 = bmid + db

    return l1, l2, b1, b2

In [ ]:
def plot_patch(hdr1, map1, map2, l1,l2,b1,b2,lmid,bmid,fs=26, RMmax=10):

    fig = plt.figure(figsize=(24, 7))
    cmap = mpl.colormaps.get_cmap('RdBu_r')
    cmap.set_bad(color='grey')

    wcs = WCS(hdr)

    #RMmax = 10

    ax1 = fig.add_subplot(131, projection=wcs)
    ax2 = fig.add_subplot(132, projection=wcs)
    ax3 = fig.add_subplot(133, projection=wcs)

    axs = [ax1, ax2, ax3]

    plt.subplots_adjust(left=0.05, bottom=0.15, right=0.85, top=0.99, wspace=0.3)

    im1 = ax1.imshow(map1, origin='lower', cmap=cmap, vmin=-RMmax, vmax=RMmax)
    im2 = ax2.imshow(map2, origin='lower', cmap=cmap, vmin=-RMmax, vmax=RMmax)

    cat_coords = SkyCoord(l_arr, b_arr, frame=Galactic, unit="deg")
    ax3.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')

    ims = [im1, im2]

    #label = 'Hutschenreuter RM'

    cx = SkyCoord([l2,l1], [bmid,bmid], frame=Galactic, unit="deg")
    cy = SkyCoord([lmid,lmid], [b1,b2], frame=Galactic, unit="deg")

    for i in range(0,3):
        axs[i].coords.grid()
        axs[i].set_xlim(wcs.world_to_pixel(cx)[0])
        axs[i].set_ylim(wcs.world_to_pixel(cy)[1])
        #ax.coords[0].set_ticks(spacing=10*u.deg)
        #ax.coords[1].set_ticks(spacing=10*u.deg)
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel(' ')

        for spine in axs[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        axs[i].tick_params(axis='both', labelsize=fs, which='both', width=2, length=6)

        #cat_coords_lofar = SkyCoord(cat['l'], cat['b'], frame=Galactic, unit="deg")
    
        #ax1.scatter(wcs.world_to_pixel(cat_coords_lofar)[0], wcs.world_to_pixel(cat_coords_lofar)[1], 
        #           s=50, color='k')

        axs[i].set_ylabel('Galactic Latitude',fontsize=fs)
        axs[i].set_xlabel('Galactic Longitude',fontsize=fs)

        #cbar_ax = fig.add_axes([0.9, 0.15, 0.02, 0.8])  # [left, bottom, width, height]
        #cbar = fig.colorbar(ims[i], cax=cbar_ax, orientation='vertical')
        ##cbar.set_ticks(np.arange(-10,12,2))
        #cbar.set_label('RM (rad m$^{-2}$)', fontsize=fs, labelpad=2)
        #cbar.ax.tick_params(labelsize=fs, length=6, width=2)
        #for spine in cbar.ax.spines.values():
        #    spine.set_visible(True)
        #    spine.set_linewidth(2)
    

    #plt.savefig(f'/home/aordog/Dropbox/plots/lotss-dragons/manuscript_plots/referee_report/EG_RM_dist_{lmid}_{bmid}.png')
    #plt.savefig(f'/home/aordog/Dropbox/plots/lotss-dragons/manuscript_plots/referee_report/EG_RM_dist_{lmid}_{bmid}.pdf')

    return

## Check distributions of EG RMs:

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(12,4))

snr_thresh = 5
RMlim = 800

axs[0].hist(cat_spiceracs['rm'], bins=201,range=(-RMlim,RMlim),color='grey');
axs[0].hist(cat_spiceracs['rm'][cat_spiceracs['snr_polint']>snr_thresh], bins=201,range=(-RMlim,RMlim),
           color='k');
axs[0].set_xlim(-RMlim,RMlim)
axs[0].set_xlabel(r'RM (rad m$^{-2}$)')

axs[1].hist(cat_spiceracs['snr_polint'], bins=200,range=(0,10),color='grey');
axs[1].hist(cat_spiceracs['snr_polint'][cat_spiceracs['snr_polint']>snr_thresh], bins=200,range=(0,10),
            color='k');
axs[1].set_xlim(0,10)
axs[1].set_xlabel(r'S:N')

axs[2].hist(cat_spiceracs['rm_err'], bins=200,range=(0,20),color='grey');
axs[2].hist(cat_spiceracs['rm_err'][cat_spiceracs['snr_polint']>snr_thresh], bins=200,range=(0,20),
           color='k');
axs[2].set_xlim(0,20)
axs[2].set_xlabel(r'err$_{\text{RM}}$ (rad m$^{-2}$)')

plt.tight_layout()

plt.savefig(f'/home/aordog/Dropbox/plots/lotss-dragons/spiceracs_compare/spiceracs_filtering.png')
plt.savefig(f'/home/aordog/Dropbox/plots/lotss-dragons/spiceracs_compare/spiceracs_filtering.pdf')

In [ ]:
print(len(cat_spiceracs['rm']))
print(len(cat_spiceracs['rm'][cat_spiceracs['snr_polint']>snr_thresh]))

In [ ]:
def plot_patch2(hdr1, map1, map2, map3, l1,l2,b1,b2,lmid,bmid,fs=26, RMmax=10):

    fig = plt.figure(figsize=(24, 20))
    cmap = mpl.colormaps.get_cmap('RdBu_r')
    cmap.set_bad(color='grey')

    wcs = WCS(hdr)

    idx_D = hp.pixelfunc.ang2pix(128, lmid, bmid, lonlat=True)
    idx_L = hp.pixelfunc.ang2pix(64, lmid, bmid, lonlat=True)
    idx_H = hp.pixelfunc.ang2pix(256, lmid, bmid, lonlat=True)

    #RMmax = 10

    axd1 = fig.add_subplot(331, projection=wcs)
    axd2 = fig.add_subplot(334, projection=wcs)

    axl1 = fig.add_subplot(332, projection=wcs)
    axl2 = fig.add_subplot(335, projection=wcs)

    axh1 = fig.add_subplot(333, projection=wcs)
    axh2 = fig.add_subplot(336, projection=wcs)

    axsp = fig.add_subplot(313)

    plt.subplots_adjust(left=0.1, bottom=0.05, right=0.95, top=0.99, wspace=0.1, hspace=0.12)

    axsp.plot(fd_dragons, dragons_cube_gal[:,idx_D], linestyle='dashed',color='k',linewidth=2)
    axsp.plot(fd_hbn, hbn_cube_gal[:,idx_H]*3,color='C0',linewidth=2)
    axsp2 = axsp.twinx()
    axsp2.plot(fd_lotss, lotss_cube_gal_S[:,idx_L]*1e3, color='grey',linewidth=2)

    axs = [axd1, axd2, axl1, axl2, axh1, axh2, axsp, axsp2]

    print(len(rm_arr_SP2))
    for j in range(0,len(rm_arr_SP2)):
        axsp.axvline(x=rm_arr_SP2[j], color='blue')
        print(l_arr_SP2[j],b_arr_SP2[j])

    print(len(rm_arr_H2))
    for j in range(0,len(rm_arr_H2)):
        axsp.axvline(x=rm_arr_H2[j], color='red')
        print(l_arr_H2[j],b_arr_H2[j])
    
    axsp.set_xlim(-200,200)
    axsp.set_ylim(0,)
    axsp2.set_ylim(0,)

    
    for i in range(0,2):
        im1 = axs[i].imshow(map1, origin='lower', cmap=cmap, vmin=-RMmax, vmax=RMmax)
    for i in range(2,4):
        im2 = axs[i].imshow(map2, origin='lower', cmap=cmap, vmin=-RMmax, vmax=RMmax)
    for i in range(4,6):
        im3 = axs[i].imshow(map3, origin='lower', cmap=cmap, vmin=-RMmax, vmax=RMmax)

    cat_coords = SkyCoord(l_arr_SP, b_arr_SP, frame=Galactic, unit="deg")
    axd1.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_SP, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')
    axl1.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_SP, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')
    axh1.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_SP, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')

    cat_coords = SkyCoord(l_arr_H, b_arr_H, frame=Galactic, unit="deg")
    axd2.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_H, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')
    axl2.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_H, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')
    axh2.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_H, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='k')

    # plot the 1-degree pixel sources
    cat_coords = SkyCoord(l_arr_SP2, b_arr_SP2, frame=Galactic, unit="deg")
    axd1.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_SP2, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='gold')
    axl1.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_SP2, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='gold')
    axh1.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_SP2, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='gold')

    cat_coords = SkyCoord(l_arr_H2, b_arr_H2, frame=Galactic, unit="deg")
    axd2.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_H2, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='gold')
    axl2.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_H2, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='gold')
    axh2.scatter(wcs.world_to_pixel(cat_coords)[0], wcs.world_to_pixel(cat_coords)[1], 
                   s=100, c=rm_arr_H2, cmap=cmap, vmin=-RMmax, vmax=RMmax, edgecolor='gold')

    ims = [im1, im2, im3]

    cx = SkyCoord([l2,l1], [bmid,bmid], frame=Galactic, unit="deg")
    cy = SkyCoord([lmid,lmid], [b1,b2], frame=Galactic, unit="deg")

    for i in range(0,8):
        if i < 6:
            axs[i].coords.grid()
            axs[i].set_xlim(wcs.world_to_pixel(cx)[0])
            axs[i].set_ylim(wcs.world_to_pixel(cy)[1])
            axs[i].set_xlabel(' ')
            axs[i].set_ylabel(' ')

        for spine in axs[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        axs[i].tick_params(axis='both', labelsize=fs, which='both', width=2, length=6)

    axd1.set_ylabel('Galactic Latitude',fontsize=fs)
    axd2.set_ylabel('Galactic Latitude',fontsize=fs)
    axd2.set_xlabel('Galactic Longitude',fontsize=fs)
    axl2.set_xlabel('Galactic Longitude',fontsize=fs)
    axsp.set_xlabel(r'FD and RM (rad m$^{-2}$)',fontsize=fs)

    plt.savefig(f'/home/aordog/Dropbox/plots/lotss-dragons/spiceracs_compare/spiceracs_compare_{lmid}_{bmid}_v2.png')
    plt.savefig(f'/home/aordog/Dropbox/plots/lotss-dragons/spiceracs_compare/spiceracs_compare_{lmid}_{bmid}_v2.pdf')

In [ ]:
#lmid =  90  90 120  135  131.5  103.4
#bmid = -30 -25 -35  -37  -37.2  -25.9

#lmid = 120
bmid = -50

l_list = [140,135,130,125,120,115,110,105,100,95,90,85]

for lmid in l_list:

    map_lotss, map_dragons, map_hbn, hdr, l1, l2, b1, b2 = reproject_patch(m1_lotss_gal_S, m1_dragons, m1_hbn, lmid, bmid, 2)

    l1_pix, l2_pix, b1_pix, b2_pix = points_in_pixel(lmid, bmid, 0.5)
    
    idx = np.where((cat_spiceracs['l']>l1) & (cat_spiceracs['l']<l2) & (cat_spiceracs['b']>b1) & (cat_spiceracs['b']<b2) & (cat_spiceracs['snr_polint']>5) )
    idx_pix = np.where((cat_spiceracs['l']>l1_pix) & (cat_spiceracs['l']<l2_pix) & (cat_spiceracs['b']>b1_pix) & (cat_spiceracs['b']<b2_pix) & (cat_spiceracs['snr_polint']>5) )
    l_arr_SP = cat_spiceracs['l'][idx]
    b_arr_SP = cat_spiceracs['b'][idx]
    rm_arr_SP = cat_spiceracs['rm'][idx]
    l_arr_SP2 = cat_spiceracs['l'][idx_pix]
    b_arr_SP2 = cat_spiceracs['b'][idx_pix]
    rm_arr_SP2 = cat_spiceracs['rm'][idx_pix]
    
    idx = np.where((cat_hutschen['l']>l1) & (cat_hutschen['l']<l2) & (cat_hutschen['b']>b1) & (cat_hutschen['b']<b2))
    idx_pix = np.where((cat_hutschen['l']>l1_pix) & (cat_hutschen['l']<l2_pix) & (cat_hutschen['b']>b1_pix) & (cat_hutschen['b']<b2_pix))
    l_arr_H = cat_hutschen['l'][idx]
    b_arr_H = cat_hutschen['b'][idx]
    rm_arr_H = cat_hutschen['rm'][idx]
    l_arr_H2 = cat_hutschen['l'][idx_pix]
    b_arr_H2 = cat_hutschen['b'][idx_pix]
    rm_arr_H2 = cat_hutschen['rm'][idx_pix]
    
    plot_patch2(hdr, map_dragons, map_lotss, map_hbn, l1,l2,b1,b2,lmid,bmid,fs=16, RMmax=20)


In [ ]:
print(spiceracs[1].columns)

In [ ]:
this = spiceracs[1].data['ra']

In [ ]:
this.shape